# Interactive Filter

## What you'll learn

- Load artifacts and their metrics into an InteractiveFilter
- Explore metric distributions with wide and tidy DataFrames
- Set filter criteria and inspect pass rates
- Visualize distributions with threshold overlay plots
- Commit the filter result as a pipeline step

**InteractiveFilter** lets you explore metric distributions before choosing
thresholds. Once you have selected criteria, `.commit()` records the filter
as a pipeline step with provenance.

**Prerequisites:** [Metrics and Filtering](../02-pipeline-design/03-metrics-and-filtering.ipynb),
[Exploring Results](../01-getting-started/02-exploring-results.ipynb).  
**Estimated time:** 15 minutes  
**GPU required:** No.

In [ ]:
from __future__ import annotations

from artisan.operations.curator import InteractiveFilter
from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager, Runner
from artisan.utils import tutorial_setup
from artisan.visualization import build_macro_graph, build_micro_graph, inspect_pipeline

In [ ]:
env = tutorial_setup("interactive_filter")

## Build a pipeline with metrics

Generate data, transform it, and compute the metrics that InteractiveFilter
will load.

In [ ]:
pipeline = PipelineManager.create(
    name="interactive_filter_tutorial",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 8, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)
pipeline.finalize()
inspect_pipeline(env.delta_root)

## Load artifacts into the interactive filter

Create an `InteractiveFilter` pointed at the delta root, then call
`.load()` to pull in artifacts and their derived metrics. The
`step_numbers` argument controls which primary artifacts to load —
here we load the transformed datasets from step 1. Supply `pipeline_run_id`
to select this run explicitly; omission selects the newest run.

In [ ]:
filt = InteractiveFilter(env.delta_root)
filt.load(step_numbers=[1], pipeline_run_id=pipeline.config.pipeline_run_id)

## Explore metric distributions

Inspect the loaded metrics as a wide DataFrame, then as a tidy DataFrame:

In [ ]:
filt.wide_df

In [ ]:
filt.tidy_df

`wide_df` has one row per artifact and one column per metric. Nested keys
use dot paths such as `distribution.median` and `summary.cv`.

`tidy_df` has one row per artifact–metric pair, with columns `artifact_id`,
`step_number`, `step_name`, `metric_name`, `metric_value`, and `metric_compound`.
Use this form for grouping and plotting.

## Set filter criteria

Each criterion specifies a metric column name,
a comparison operator (`gt`, `ge`, `lt`, `le`, `eq`, `ne`), and a
threshold value. The metric name must match a column in `wide_df`.

In [ ]:
filt.set_criteria(
    [
        {"metric": "distribution.median", "operator": "gt", "value": 0.3},
        {"metric": "summary.cv", "operator": "lt", "value": 0.8},
    ]
)

## Inspect pass rates

Call `.summary()` to inspect the criteria and cumulative results:

In [ ]:
filt.summary()

The criteria table shows each threshold, pass count, pass rate, and metric
statistics (min, mean, max). The funnel table shows how many artifacts remain
after each successive criterion. Use it to find the most restrictive criterion.

## Visualize distributions

`.plot()` draws a histogram for each criterion with a red dashed threshold line.

In [ ]:
filt.plot()

## Iterate on thresholds

If the pass rates aren't what you want, adjust the criteria and
recheck. The loaded metrics let you compare thresholds without rerunning
the pipeline.

In [ ]:
filt.set_criteria(
    [
        {"metric": "distribution.median", "operator": "gt", "value": 0.4},
    ]
)
filt.summary()

## Commit the filter as a pipeline step

When you're satisfied with the thresholds, call `.commit()` to write
the filter as a pipeline step. This creates step, execution, and
provenance records in the delta store — making the filter result
available to downstream steps via `result.output("passthrough")`.

In [ ]:
result = filt.commit()
print(f"Committed as step {result.step_number}")
print(f"  {result.succeeded_count} / {result.total_count} artifacts passed")

## Inspect the pipeline after commit

The committed filter now appears in the pipeline overview and the
macro graph.

In [ ]:
inspect_pipeline(env.delta_root)

In [ ]:
build_macro_graph(env.delta_root)

The macro graph includes the committed filter step. Its `passthrough` output
contains the artifacts selected by your thresholds.

In [ ]:
build_micro_graph(env.delta_root)

## Next steps

- [Timing Analysis](04-timing-analysis.ipynb) — Profile step and execution performance
- [Provenance Graphs](01-provenance-graphs.ipynb) — Visualize the filter step in context
- [Metrics and Filtering](../02-pipeline-design/03-metrics-and-filtering.ipynb) — Automated filtering with upfront criteria
- [Provenance System](../../concepts/provenance-system.md) — How commit records the filter step with full provenance
- [Writing Curator Operations](../../how-to-guides/writing-curator-operations.md) — Build your own filter and merge operations